In [3]:
import Pkg; Pkg.activate(@__DIR__); Pkg.add("ForwardDiff"); Pkg.instantiate()

  Activating project at `/workspaces/lecture-notebooks/Lecture 14`
    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed ForwardDiff ─ v1.2.0
    Updating `/workspaces/lecture-notebooks/Lecture 14/Project.toml`
  [f6369f11] + ForwardDiff v1.2.0
    Updating `/workspaces/lecture-notebooks/Lecture 14/Manifest.toml`
  [bbf7d656] + CommonSubexpressions v0.3.1
  [163ba53b] + DiffResults v1.1.0
  [b552c78f] + DiffRules v1.15.1
  [ffbed154] + DocStringExtensions v0.9.5
  [f6369f11] + ForwardDiff v1.2.0
  [92d709cd] + IrrationalConstants v0.2.4
  [692b3bcd] + JLLWrappers v1.7.1
  [2ab3a3ac] + LogExpFunctions v0.3.29
  [1914dd2f] + MacroTools v0.5.16
  [77ba4419] + NaNMath v1.1.3
  [21216c6a] + Preferences v1.5.0
  [276daf66] + SpecialFunctions v2.5.1
  [1e83bf80] + StaticArraysCore v1.4.3
  [efe28fd5] + OpenSpecFun_jll v0.5.6+0
  [56f22d72] + Artifacts
  [ade2ca70] + Dates
  [8f399da3] + Libdl
  [37e2e46d] + LinearAlgebra
  [de0858da] + Print

In [4]:
using LinearAlgebra
using ForwardDiff

In [5]:
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

hat (generic function with 1 method)

In [6]:
function L(q)
    s = q[1]
    v = q[2:4]
    L = [s    -v';
         v  s*I+hat(v)]
    return L
end

L (generic function with 1 method)

In [22]:
function R(q)
    s = q[1]
    v = q[2:4]
    R = [s    -v';
         v  s*I-hat(v)]
    return R
end

R (generic function with 1 method)

In [7]:
T = Diagonal([1; -ones(3)])
H = [zeros(1,3); I];

In [8]:
function G(q)
    G = L(q)*H
end

function Q(q)
    return H'*(R(q)'*L(q))*H
end

Q (generic function with 1 method)

In [9]:
J = Diagonal([1; 2; 3])
h = 0.1

0.1

In [10]:
#initial conditions
Q0 = Array(I(3))
q0 = [1; 0; 0; 0]
ω0 = randn(3)
x0 = [vec(Q0); ω0]
x0q = [q0; ω0]

7-element Vector{Float64}:
  1.0
  0.0
  0.0
  0.0
  1.1923584515092016
 -1.1641382964094658
 -1.6933678834554045

In [11]:
#dynamics
function dynamics(x)
    Q = reshape(x[1:9],3,3)
    ω = x[10:12]
    
    Q̇ = Q*hat(ω)
    ω̇ = -J\(hat(ω)*J*ω)

    ẋ = [vec(Q̇); ω̇]
end

dynamics (generic function with 1 method)

In [12]:
function rkstep(x)
    f1 = dynamics(x)
    f2 = dynamics(x + 0.5*h*f1)
    f3 = dynamics(x + 0.5*h*f2)
    f4 = dynamics(x + h*f3)
    xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
    return xn
end

rkstep (generic function with 1 method)

In [13]:
xk = x0
for k = 1:10000
    xk = rkstep(xk)
end

In [14]:
Qk = reshape(xk[1:9],3,3)

3×3 Matrix{Float64}:
 -0.873726    0.453222   0.0491273
 -0.454199   -0.867135  -0.128752
 -0.0162131  -0.149971   0.985024

In [15]:
Qk'*Qk

3×3 Matrix{Float64}:
  0.969956      0.000291411  -0.000415063
  0.000291411   0.979824     -0.0138138
 -0.000415063  -0.0138138     0.989263

In [16]:
#quaternion dynamics
function qdynamics(x)
    q = x[1:4]
    ω = x[5:7]
    
    q̇ = 0.5*L(q)*H*ω
    ω̇ = -J\(hat(ω)*J*ω)

    ẋ = [q̇; ω̇]
end

qdynamics (generic function with 1 method)

In [17]:
function qrkstep(x)
    f1 = qdynamics(x)
    f2 = qdynamics(x + 0.5*h*f1)
    f3 = qdynamics(x + 0.5*h*f2)
    f4 = qdynamics(x + h*f3)
    xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
    xn[1:4] .= xn[1:4]./norm(xn[1:4])
    return xn
end

qrkstep (generic function with 1 method)

In [18]:
xkq = x0q
for k = 1:10000
    xkq = qrkstep(xkq)
end

In [19]:
qk = xkq[1:4]

4-element Vector{Float64}:
  0.21254138173502599
  0.005780584701348081
  0.07566566399731517
 -0.9742009306003162

In [20]:
norm(qk)

1.0

In [23]:
Q(qk)'*Q(qk)

3×3 Matrix{Float64}:
 1.0          2.60209e-17  2.08167e-17
 2.60209e-17  1.0          5.55112e-17
 2.08167e-17  5.55112e-17  1.0